# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates step-by-step how to explore and analyze the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load the metadata structure and data records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Croissant schema URL for FAIR^2 dataset:
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata.to_json()
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print("Dataset metadata summary:")
pprint.pprint(metadata)  # Pretty print a summary of the metadata

## 2. Data Overview

### Record Sets
List the available record sets and their `@id`s.

Inspect the fields and columns available in each record set.

In [ ]:
# List all record sets in the metadata and show their fields (referenced by `@id`)
record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record sets.")

recordset_overview = []
for rs in record_sets:
    print("---")
    print(f"Record Set: {rs.name}")
    print(f"  @id: {rs.id}")
    # Fields and columns for each RecordSet
    print("  Fields and columns:")
    for field in rs.fields:
        print(f"    - Field: {field.name} (@id: {field.id}, dataType: {field.data_type})")
        if hasattr(field, "column"):
            # Each field might point to a column
            col = field.column
            print(f"      (column: {col.name}, column @id: {col.id})")
        else:
            print("      (no column)")
    recordset_overview.append({
        "name": rs.name,
        "id": rs.id,
        "fields": [(field.name, field.id, field.data_type) for field in rs.fields]
    })

#### Example records from a record set

Let's display a few records from the main protein abundance table (replace with the appropriate record set `@id` from above).

In [ ]:
# Select the main record set by `@id`
# Replace the value below with the appropriate @id of the primary data table from the overview above.
main_record_set_id = record_sets[0].id if record_sets else None  # Fallback if only one
print(f"Showing the first 3 records from record set {main_record_set_id}\n")
for idx, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(record)
    if idx >= 2:  # Show only first 3
        break

## 3. Data Extraction

Let's load data from all main record sets into Pandas DataFrames.

Use the `@id` values of your desired record sets as shown in the previous overview cell.

In [ ]:
# Record set ids we wish to extract
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Extracting records for record set: {record_set_id}")
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"  Shape: {df.shape}")
    if len(df.columns) > 0:
        print(f"  Columns: {list(df.columns)}\n")

### Preview table columns and first few rows for the main record set

In [ ]:
# Let's work with the first/main record set
main_rs_df = dataframes[main_record_set_id]
print(f"Columns for {main_record_set_id}:")
print(main_rs_df.columns.tolist())
main_rs_df.head()

## 4. Exploratory Data Analysis (EDA)

We'll perform some common processing: filter by a numeric column (e.g., peptide count or abundance), normalize a column, and group by a categorical column if available.

- Replace `numeric_field_id` and `group_field_id` below by looking up field IDs/names in your previous overview.

In [ ]:
# Pick your numeric field and group field by @id or column name.
df = main_rs_df
# Example: Let's try to infer a numeric column (e.g., peptide count, mw, or abundance) – adjust as needed.
possible_numeric_cols = df.select_dtypes(include='number').columns.tolist()
if not possible_numeric_cols:
    # Fallback if all are strings, try to convert what looks like numeric
    for col in df.columns:
        try:
            df[col + "_float"] = pd.to_numeric(df[col], errors='coerce')
        except Exception:
            continue
    possible_numeric_cols = df.select_dtypes(include='number').columns.tolist()

if possible_numeric_cols:
    numeric_field = possible_numeric_cols[0]
    print(f"Using numeric field: {numeric_field}")
else:
    raise ValueError("No numeric columns found in the main record set DataFrame.")

# Filter records where the numeric field > threshold
threshold = df[numeric_field].mean() if numeric_field else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records (>{threshold:.2f}) for {numeric_field}: {filtered_df.shape[0]} rows")

# Normalize this numeric field
filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Try grouping by a likely categorical field (e.g., 'accession', 'description', or something else categorical)
possible_cat_cols = df.select_dtypes(include='object').columns.tolist()
group_field = None
# Exclude too-unique fields (ID fields)
for col in possible_cat_cols:
    if df[col].nunique() < (0.2 * len(df)):
        group_field = col
        break
if group_field:
    print(f"\nGrouping by: {group_field}\n")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization

Let's visualize the distribution of the numeric feature and, if possible, the group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# If group_field was found, plot the top mean values by group
if 'grouped_df' in locals() and group_field is not None:
    plt.figure(figsize=(10, 4))
    top_groups = grouped_df.sort_values(by=numeric_field, ascending=False).head(20)
    sns.barplot(y=group_field, x=numeric_field, data=top_groups)
    plt.title(f"Top 20 mean {numeric_field} by {group_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We loaded a mass-spectrometry-based extracellular vesicle protein dataset using the Croissant format and the `mlcroissant` library.
- We explored available record sets, fields and their structure via `@id` references.
- We extracted, filtered, normalized, and visualized key columns of the dataset.

This notebook can be extended to more sophisticated downstream analyses, such as machine learning, feature engineering, or biological/clinical interpretation as suitable for the domain.